# Hierarchical Spectroscopy: Breaking the PSD Degeneracy

Demo 04 showed hierarchical PSD recovery from **photometry** (5 SDSS bands).
While the population model recovers $\sigma_{\rm PS}$ reasonably well, the
PSD timescale $\tau_{\rm PS}$ remains poorly constrained --- 5 broadband fluxes
simply lack the spectral resolution to pin down burstiness timescales.

**Spectroscopy changes everything.** With ~200 pixels per galaxy (~40x more
data points), spectral features like the 4000-A break, Balmer absorption
lines, and UV slope encode star formation history at multiple timescales.
This notebook demonstrates that hierarchical spectroscopic inference breaks
the $\sigma$--$\tau$ degeneracy that photometry alone cannot resolve.

**Outline:**
1. Generate a population of 10 mock galaxies with shared PSD
2. Show example spectra and their information content
3. Individual EVI fits (PSD free per galaxy)
4. Hierarchical EVI on spectroscopy
5. Key result: spectroscopy vs photometry PSD posteriors
6. SFH recovery from the hierarchical posterior
7. Posterior width scaling with population size ($\propto 1/\sqrt{N}$)

In [ ]:
import os
import sys
import time
import warnings

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

jax.config.update("jax_enable_x64", True)
warnings.filterwarnings("ignore", category=FutureWarning)

from tengri import (
    Fitter,
    Fixed,
    HierarchicalFitter,
    Model,
    Observation,
    ParamSpec,
    Photometry,
    SpectroscopyConfig,
    Uniform,
    load_ssp_data,
)

# Notebook path handling
try:
    _nb_dir = os.path.dirname(os.path.abspath(__file__))
    sys.path.insert(0, os.path.join(_nb_dir, "..", ".."))
except NameError:
    _nb_dir = os.getcwd()
    sys.path.insert(0, os.path.join(_nb_dir, ".."))

# Change to project root so data/ paths work
if os.path.exists("data"):
    pass
elif os.path.exists(os.path.join("..", "data")):
    os.chdir("..")
elif os.path.exists(os.path.join("..", "..", "data")):
    os.chdir(os.path.join("..", ".."))
elif os.path.exists(os.path.join("..", "..", "..", "data")):
    os.chdir(os.path.join("..", "..", ".."))

FIGDIR = os.path.join("notebooks", "figures", "demonstrations")
os.makedirs(FIGDIR, exist_ok=True)

from _plot_style import (
    COLORS,
    convergence_check,
    convergence_table,
    plot_sfh,
    setup_style,
)

setup_style()

## 1. Motivation

Photometry provides only $\sim$5 data points per galaxy. Even with
hierarchical pooling across a population, the PSD timescale $\tau_{\rm PS}$
spans most of its prior (demo 04). Spectroscopy offers $\sim$40x more
constraints per galaxy through:

- **D4000 break**: age-sensitive absorption feature
- **Balmer lines** ($H\beta$, $H\gamma$, $H\delta$): trace recent SF on
  100 Myr timescales
- **UV slope**: sensitive to current-to-past SFR ratio
- **Metal absorption lines**: break age-metallicity degeneracy

The hierarchical model shares PSD hyperparameters
$\phi = (\sigma_{\rm PS}, \tau_{\rm PS})$ across the population while
fitting per-galaxy latent variables: GP field $\xi_i$ (128 dims) and
physical parameters $\theta_i$ (8 dims). Total dimensionality for
$N$ galaxies: $D = N \times 136 + 2$ shared.

## 2. Setup: Population and Model

In [ ]:
# Load SSP data
ssp_data = load_ssp_data("data/ssp_prsc_miles_chabrier_wNE_logGasU-3.0_logGasZ0.0.h5")

# Observation configuration
WAVE_OBS = jnp.linspace(3800.0, 9200.0, 200)
N_PIX = len(WAVE_OBS)
FILTER_NAMES = ["sdss_u", "sdss_g", "sdss_r", "sdss_i", "sdss_z"]

# True shared PSD parameters
TRUE_SIGMA = 2.0
TRUE_TAU = 20.0  # Myr
N_GAL = 10
SPEC_SNR = 30.0
PHOT_SNR = 20.0

print(
    f"Spectroscopy: {N_PIX} pixels, {float(WAVE_OBS[0]):.0f}"
    f"--{float(WAVE_OBS[-1]):.0f} A ({SPEC_SNR:.0f} SNR)"
)
print(f"Photometry: {len(FILTER_NAMES)} SDSS bands ({PHOT_SNR:.0f} SNR)")
print(f"Population: N = {N_GAL}, true sigma = {TRUE_SIGMA}, true tau = {TRUE_TAU} Myr")

In [ ]:
# Model factory for hierarchical inference — uses Observation API
def model_factory(psd_sigma=1.0, psd_tau_myr=50.0):
    """Create a Model with fixed PSD, called by HierarchicalFitter.

    Uses the Observation API for both photometry and spectroscopy
    configuration. Star-forming prior with positive skew and
    peak_lbt_gyr centered at 3.0 Gyr.
    """
    spec = ParamSpec(
        sfh_tsnorm_log_peak_sfr=Uniform(-1.0, 2.5),
        sfh_tsnorm_peak_lbt_gyr=Uniform(0.5, 12.0),
        sfh_tsnorm_width_gyr=Uniform(0.3, 5.0),
        sfh_tsnorm_skew=Uniform(-3.0, 3.0),
        sfh_tsnorm_trunc=Uniform(1.0, 10.0),
        sfh_field_psd_sigma=Fixed(psd_sigma),
        sfh_field_psd_tau_myr=Fixed(psd_tau_myr),
        met_logzsol=Uniform(-2.0, 0.2),
        dust_tau_bc=Uniform(0.0, 2.0),
        dust_tau_diff=Uniform(0.0, 1.5),
        dust_slope=Fixed(-0.7),
        redshift=Fixed(0.1),
        mean_sfh_type=["tsnorm", "field"],
        n_grid=128,
    )
    obs = Observation(
        photometry=Photometry.from_names(FILTER_NAMES),
        spectroscopy=SpectroscopyConfig(wave_obs=WAVE_OBS),
    )
    return Model(spec, ssp_data, observation=obs)

In [ ]:
# Generate mock population with shared PSD
print(f"Generating {N_GAL} mock galaxies...")
key = jax.random.PRNGKey(42)
model_gen = model_factory(psd_sigma=TRUE_SIGMA, psd_tau_myr=TRUE_TAU)

galaxies_spec = []
galaxies_phot = []
true_params_all = []
true_spectra = []

for i in range(N_GAL):
    k = jax.random.fold_in(key, i)
    params = model_gen.spec.sample(k)

    # Spectroscopic mock
    mock_s = model_gen.mock_spectrum(params, WAVE_OBS, snr=SPEC_SNR, key=jax.random.fold_in(k, 1))
    galaxies_spec.append({"flux_obs": mock_s.flux_obs, "noise": mock_s.noise})
    true_spectra.append(np.array(mock_s.flux_true))

    # Photometric mock (same galaxy, for later comparison)
    mock_p = model_gen.mock(params, snr=PHOT_SNR, key=jax.random.fold_in(k, 2))
    galaxies_phot.append({"flux_obs": mock_p.flux_obs, "noise": mock_p.noise})

    true_params_all.append(params)

print(f"  Per-galaxy: D = {model_gen.spec.n_free} physical + 128 GP")
print(
    f"  Total (hierarchical): D = {N_GAL} x {model_gen.spec.n_free + 128}"
    f" + 2 shared = {N_GAL * (model_gen.spec.n_free + 128) + 2}"
)

## 3. Mock Spectra

Each galaxy has a unique SFH drawn from the shared PSD prior. The
stochastic GP fluctuations create diverse spectral shapes that encode
different recent star formation histories.

In [ ]:
# --- FIGURE 1: 4 example spectra ---
fig, axes = plt.subplots(2, 2, figsize=(12, 7))
wave_np = np.array(WAVE_OBS)

for ax, idx in zip(axes.ravel(), range(4)):
    flux_obs = np.array(galaxies_spec[idx]["flux_obs"])
    noise = np.array(galaxies_spec[idx]["noise"])
    flux_true = true_spectra[idx]

    ax.fill_between(
        wave_np,
        flux_obs - noise,
        flux_obs + noise,
        alpha=0.25,
        color=COLORS["data"],
        label=r"$\pm 1\sigma$",
    )
    ax.plot(wave_np, flux_obs, color=COLORS["data"], lw=0.4, alpha=0.6)
    ax.plot(
        wave_np,
        flux_true,
        color=COLORS["truth"],
        lw=1.2,
        label="Truth",
    )
    ax.set_xlabel(r"$\lambda_{\rm obs}$ [$\AA$]")
    if idx % 2 == 0:
        ax.set_ylabel(r"$f_\nu$")
    ax.set_title(f"Galaxy {idx}", fontsize=10)
    if idx == 0:
        ax.legend(fontsize=8)

fig.suptitle(
    f"Mock spectra: {N_PIX} pixels, SNR = {SPEC_SNR:.0f}, "
    rf"shared $\sigma$ = {TRUE_SIGMA}, $\tau$ = {TRUE_TAU} Myr",
    fontsize=11,
)
fig.tight_layout()
plt.savefig(
    os.path.join(FIGDIR, "15_fig01_example_spectra.png"),
    dpi=150,
    bbox_inches="tight",
)
plt.show()

## 4. Individual EVI Fits

First, fit each galaxy independently with PSD parameters **free**.
This establishes the per-galaxy constraining power before hierarchical
pooling. As in demo 04, $\sigma_{\rm PS}$ is roughly constrained but
$\tau_{\rm PS}$ spans much of its prior.

In [ ]:
# Individual native_geovi fits with FREE PSD
spec_free = ParamSpec(
    sfh_tsnorm_log_peak_sfr=Uniform(-1.0, 2.5),
    sfh_tsnorm_peak_lbt_gyr=Uniform(0.5, 12.0),
    sfh_tsnorm_width_gyr=Uniform(0.3, 5.0),
    sfh_tsnorm_skew=Uniform(-3.0, 3.0),
    sfh_tsnorm_trunc=Uniform(1.0, 10.0),
    sfh_field_psd_sigma=Uniform(0.1, 4.0),
    sfh_field_psd_tau_myr=Uniform(1.0, 300.0),
    met_logzsol=Uniform(-2.0, 0.2),
    dust_tau_bc=Uniform(0.0, 2.0),
    dust_tau_diff=Uniform(0.0, 1.5),
    dust_slope=Fixed(-0.7),
    redshift=Fixed(0.1),
    mean_sfh_type=["tsnorm", "field"],
    n_grid=128,
)
obs_spec = Observation(
    spectroscopy=SpectroscopyConfig(wave_obs=WAVE_OBS),
    photometry=Photometry.from_names(FILTER_NAMES),
)
model_free = Model(spec_free, ssp_data, observation=obs_spec)

individual_results = []
print("Fitting 4 galaxies individually (PSD free)...")
for i in range(min(4, N_GAL)):
    t0 = time.perf_counter()
    fitter_i = Fitter(
        model_free,
        galaxies_spec[i]["flux_obs"],
        galaxies_spec[i]["noise"],
        data_type="spectroscopy",
    )
    res_i = fitter_i.run(
        "native_geovi",
        n_iterations=15,
        n_samples=6,
        n_seeds=3,
        n_posterior_samples=500,
        verbose=False,
    )
    dt = time.perf_counter() - t0
    individual_results.append(res_i)
    sig_med = float(jnp.median(res_i.samples["sfh_field_psd_sigma"]))
    tau_med = float(jnp.median(res_i.samples["sfh_field_psd_tau_myr"]))
    print(f"  Galaxy {i}: sigma = {sig_med:.2f}, tau = {tau_med:.0f} Myr  ({dt:.1f}s)")

In [ ]:
# --- FIGURE 2: Individual PSD posteriors (wide, overlapping) ---
fig, (ax_sig, ax_tau) = plt.subplots(1, 2, figsize=(10, 4))

for i, res in enumerate(individual_results):
    sig_s = np.array(res.samples["sfh_field_psd_sigma"])
    tau_s = np.array(res.samples["sfh_field_psd_tau_myr"])
    ax_sig.hist(sig_s, bins=30, alpha=0.35, density=True, label=f"Gal {i}")
    ax_tau.hist(tau_s, bins=30, alpha=0.35, density=True, label=f"Gal {i}")

ax_sig.axvline(
    TRUE_SIGMA,
    color=COLORS["truth"],
    lw=2,
    ls="--",
    label="Truth",
)
ax_tau.axvline(
    TRUE_TAU,
    color=COLORS["truth"],
    lw=2,
    ls="--",
    label="Truth",
)
ax_sig.set_xlabel(r"$\sigma_{\rm PS}$")
ax_tau.set_xlabel(r"$\tau_{\rm PS}$ [Myr]")
ax_sig.set_ylabel("Density")
ax_sig.legend(fontsize=7)
ax_tau.legend(fontsize=7)
ax_sig.set_title(r"$\sigma_{\rm PS}$: roughly constrained")
ax_tau.set_title(r"$\tau_{\rm PS}$: nearly unconstrained")
fig.tight_layout()
plt.savefig(
    os.path.join(FIGDIR, "15_fig02_individual_posteriors.png"),
    dpi=150,
    bbox_inches="tight",
)
plt.show()

## 5. Hierarchical EVI: Spectroscopy

The hierarchical model shares $\sigma_{\rm PS}$ and $\tau_{\rm PS}$ across
all $N$ galaxies while allowing each galaxy its own physical parameters and
GP field. This pools information about the burstiness timescale, tightening
both PSD parameters.

In [ ]:
# --- Hierarchical EVI on spectroscopic data ---
print(f"\nHierarchical fit: {N_GAL} galaxies (spectroscopy)...")
t0 = time.perf_counter()
hfitter_spec = HierarchicalFitter(
    model_factory,
    galaxies_spec,
    psd_sigma_prior=(0.1, 4.0),
    psd_tau_prior=(1.0, 300.0),
    data_type="spectroscopy",
)
result_hier_spec = hfitter_spec.run(
    "evi",
    n_iterations=50,
    n_samples=6,
    n_posterior_samples=500,
    n_seeds=10,
    verbose=False,
    key=jax.random.PRNGKey(0),
)
t_hier_spec = time.perf_counter() - t0

sig_spec = np.array(result_hier_spec.shared_samples["psd_sigma"])
tau_spec = np.array(result_hier_spec.shared_samples["psd_tau_myr"])
print(
    f"  sigma = {np.median(sig_spec):.2f} "
    f"[{np.percentile(sig_spec, 16):.2f}, "
    f"{np.percentile(sig_spec, 84):.2f}]"
)
print(
    f"  tau   = {np.median(tau_spec):.0f} "
    f"[{np.percentile(tau_spec, 16):.0f}, "
    f"{np.percentile(tau_spec, 84):.0f}] Myr"
)
print(f"  Wall time: {t_hier_spec:.1f}s")

In [ ]:
# --- Posterior predictive spectra for 4 example galaxies ---
fig, axes = plt.subplots(2, 2, figsize=(12, 7))

for ax, idx in zip(axes.ravel(), range(min(4, N_GAL))):
    flux_obs = np.array(galaxies_spec[idx]["flux_obs"])
    noise = np.array(galaxies_spec[idx]["noise"])
    flux_true = true_spectra[idx]

    # Posterior draws from individual samples (if available)
    spec_draws = []
    if result_hier_spec.individual_samples is not None:
        ind_samp = result_hier_spec.individual_samples[idx]
        n_draw = min(50, len(next(iter(ind_samp.values()))))
        for j in range(n_draw):
            draw = {k: v[j] for k, v in ind_samp.items()}
            draw["sfh_field_psd_sigma"] = TRUE_SIGMA
            draw["sfh_field_psd_tau_myr"] = TRUE_TAU
            try:
                pred = model_gen.predict_spectrum(draw, WAVE_OBS)
                spec_draws.append(np.array(pred))
            except Exception:
                pass

    ax.plot(wave_np, flux_obs, color=COLORS["data"], lw=0.4, alpha=0.5)
    ax.plot(wave_np, flux_true, color=COLORS["truth"], lw=1.0, label="Truth")

    if spec_draws:
        spec_arr = np.array(spec_draws)
        lo, hi = np.percentile(spec_arr, [16, 84], axis=0)
        median_pred = np.median(spec_arr, axis=0)
        ax.fill_between(
            wave_np,
            lo,
            hi,
            alpha=0.25,
            color=COLORS["geovi"],
            label="68% CI",
        )
        ax.plot(
            wave_np,
            median_pred,
            color=COLORS["geovi"],
            lw=0.8,
            label="Median",
        )
        residual = (flux_obs - median_pred) / noise
        chi2_dof = float(np.sum(residual**2) / len(wave_np))
        ax.set_title(
            f"Galaxy {idx} ($\\chi^2/\\nu = {chi2_dof:.2f}$)",
            fontsize=10,
        )
    else:
        ax.set_title(f"Galaxy {idx}", fontsize=10)

    ax.set_xlabel(r"$\lambda_{\rm obs}$ [$\AA$]")
    if idx % 2 == 0:
        ax.set_ylabel(r"$f_\nu$")
    if idx == 0:
        ax.legend(fontsize=8)

fig.suptitle("Posterior predictive spectra (hierarchical EVI)", fontsize=11)
fig.tight_layout()
plt.savefig(
    os.path.join(FIGDIR, "15_fig03_spectral_fits.png"),
    dpi=150,
    bbox_inches="tight",
)
plt.show()

## 6. Key Result: Spectroscopy vs Photometry

This is the central comparison. We run hierarchical EVI on the **same
galaxies** using photometry only, then overlay the PSD posteriors.
Spectroscopy should produce dramatically tighter constraints on both
$\sigma_{\rm PS}$ and especially $\tau_{\rm PS}$.

In [ ]:
# --- Hierarchical EVI on photometric data ---
print(f"\nHierarchical fit: {N_GAL} galaxies (photometry)...")
t0 = time.perf_counter()
hfitter_phot = HierarchicalFitter(
    model_factory,
    galaxies_phot,
    psd_sigma_prior=(0.1, 4.0),
    psd_tau_prior=(1.0, 300.0),
    data_type="photometry",
)
result_hier_phot = hfitter_phot.run(
    "evi",
    n_iterations=50,
    n_samples=6,
    n_posterior_samples=500,
    n_seeds=10,
    verbose=False,
    key=jax.random.PRNGKey(1),
)
t_hier_phot = time.perf_counter() - t0

sig_phot = np.array(result_hier_phot.shared_samples["psd_sigma"])
tau_phot = np.array(result_hier_phot.shared_samples["psd_tau_myr"])
print(
    f"  sigma = {np.median(sig_phot):.2f} "
    f"[{np.percentile(sig_phot, 16):.2f}, "
    f"{np.percentile(sig_phot, 84):.2f}]"
)
print(
    f"  tau   = {np.median(tau_phot):.0f} "
    f"[{np.percentile(tau_phot, 16):.0f}, "
    f"{np.percentile(tau_phot, 84):.0f}] Myr"
)
print(f"  Wall time: {t_hier_phot:.1f}s")

In [ ]:
# --- FIGURE 4: Spectroscopy vs photometry PSD posteriors (KEY FIGURE) ---
fig, (ax_sig, ax_tau) = plt.subplots(1, 2, figsize=(11, 4.5))

# Spectroscopy (bold)
ax_sig.hist(
    sig_spec,
    bins=40,
    alpha=0.6,
    density=True,
    color=COLORS["rt"],
    edgecolor="none",
    label=f"Spectroscopy ({N_PIX} pix)",
)
ax_sig.hist(
    sig_phot,
    bins=40,
    alpha=0.5,
    density=True,
    color=COLORS["geovi"],
    edgecolor="none",
    label=f"Photometry ({len(FILTER_NAMES)} bands)",
)
ax_sig.axvline(
    TRUE_SIGMA,
    color=COLORS["truth"],
    lw=2,
    ls="--",
    label=f"Truth = {TRUE_SIGMA}",
)

ax_tau.hist(
    tau_spec,
    bins=40,
    alpha=0.6,
    density=True,
    color=COLORS["rt"],
    edgecolor="none",
    label="Spectroscopy",
)
ax_tau.hist(
    tau_phot,
    bins=40,
    alpha=0.5,
    density=True,
    color=COLORS["geovi"],
    edgecolor="none",
    label="Photometry",
)
ax_tau.axvline(
    TRUE_TAU,
    color=COLORS["truth"],
    lw=2,
    ls="--",
    label=f"Truth = {TRUE_TAU} Myr",
)

ax_sig.set_xlabel(r"$\sigma_{\rm PS}$", fontsize=13)
ax_sig.set_ylabel("Density")
ax_sig.set_title(r"PSD amplitude $\sigma_{\rm PS}$")
ax_sig.legend(fontsize=9)

ax_tau.set_xlabel(r"$\tau_{\rm PS}$ [Myr]", fontsize=13)
ax_tau.set_ylabel("Density")
ax_tau.set_title(r"PSD timescale $\tau_{\rm PS}$")
ax_tau.legend(fontsize=9)

fig.suptitle(
    f"Hierarchical PSD Recovery (N = {N_GAL}): Spectroscopy vs Photometry",
    fontsize=12,
)
fig.tight_layout()
plt.savefig(
    os.path.join(FIGDIR, "15_fig04_spec_vs_phot_psd.png"),
    dpi=150,
    bbox_inches="tight",
)
plt.show()

# Quantitative comparison
print("\nPSD recovery summary:")
print(f"{'Data':<15} {'sigma (med [16,84])':<30} {'tau (med [16,84])':<30}")
print("-" * 75)
for label, s_arr, t_arr in [
    ("Spectroscopy", sig_spec, tau_spec),
    ("Photometry", sig_phot, tau_phot),
]:
    s_lo, s_hi = np.percentile(s_arr, [16, 84])
    t_lo, t_hi = np.percentile(t_arr, [16, 84])
    print(
        f"{label:<15} {np.median(s_arr):.2f} [{s_lo:.2f}, {s_hi:.2f}]"
        f"{'':>12} {np.median(t_arr):.0f} [{t_lo:.0f}, {t_hi:.0f}] Myr"
    )
print(f"\nTruth: sigma = {TRUE_SIGMA}, tau = {TRUE_TAU} Myr")

## 7. SFH Recovery

The hierarchical spectroscopic posterior also recovers individual galaxy
SFHs. The shared PSD prior acts as a physically-motivated regularizer,
preventing over-fitting while allowing galaxy-to-galaxy variation.

In [ ]:
# --- FIGURE 5: SFH recovery from hierarchical fit ---
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

for i, ax in enumerate(axes.flat):
    if i >= min(4, N_GAL):
        ax.set_visible(False)
        continue

    # True SFH
    sfh_true = model_gen.predict_sfh(true_params_all[i])
    t_gyr = np.array(sfh_true["t_gyr"])
    sfr_true = np.array(sfh_true["sfr_full"])
    sfr_mean_true = np.array(sfh_true["sfr_mean"])

    # Posterior SFH draws (if individual samples available)
    sfr_draws = []
    if result_hier_spec.individual_samples is not None:
        ind_samp = result_hier_spec.individual_samples[i]
        n_draw = min(50, len(next(iter(ind_samp.values()))))
        for j in range(n_draw):
            draw = {k: v[j] for k, v in ind_samp.items()}
            draw["sfh_field_psd_sigma"] = TRUE_SIGMA
            draw["sfh_field_psd_tau_myr"] = TRUE_TAU
            try:
                sfh_draw = model_gen.predict_sfh(draw)
                sfr_draws.append(np.array(sfh_draw["sfr_full"]))
            except Exception:
                pass

    if sfr_draws:
        sfr_arr = np.array(sfr_draws)
        lo, hi = np.percentile(sfr_arr, [16, 84], axis=0)
        ax.fill_between(
            t_gyr,
            lo,
            hi,
            alpha=0.25,
            color=COLORS["geovi"],
            label="68% CI",
        )
        ax.plot(
            t_gyr,
            np.median(sfr_arr, axis=0),
            color=COLORS["geovi"],
            lw=1.2,
            ls="--",
            label="Median",
        )

    ax.plot(t_gyr, sfr_true, color=COLORS["truth"], lw=1.5, label="Truth")
    ax.plot(
        t_gyr,
        sfr_mean_true,
        color=COLORS["sfh_mean"],
        lw=0.8,
        ls=":",
        alpha=0.4,
    )
    ax.set_xlim(0, 13.5)
    ax.set_xlabel("Lookback time [Gyr]")
    if i % 2 == 0:
        ax.set_ylabel(r"SFR [$M_\odot\,{\rm yr}^{-1}$]")
    ax.set_title(f"Galaxy {i}", fontsize=10)
    if i == 0:
        ax.legend(fontsize=8)

fig.suptitle(
    "SFH Recovery from Hierarchical Spectroscopic Fit",
    fontsize=11,
)
fig.tight_layout()
plt.savefig(
    os.path.join(FIGDIR, "15_fig05_sfh_recovery.png"),
    dpi=150,
    bbox_inches="tight",
)
plt.show()

## 8. Posterior Width Scaling: $\propto 1/\sqrt{N}$

The shared PSD posterior should shrink as we add more galaxies to the
population. This is the Bayesian analog of the central limit theorem:
each galaxy contributes independent information about the shared
burstiness physics.

In [ ]:
# --- Scaling experiment ---
N_VALUES = [2, 4, 6, 8, 10]
sigma_widths = []
tau_widths = []

print(
    f"\n{'N':>4s}  {'sigma_med':>9s}  {'sigma_CI':>14s}  "
    f"{'tau_med':>9s}  {'tau_CI':>14s}  {'Time':>6s}"
)
print("-" * 70)

for n_sub in N_VALUES:
    gals_sub = galaxies_spec[:n_sub]
    hf_sub = HierarchicalFitter(
        model_factory,
        gals_sub,
        psd_sigma_prior=(0.1, 4.0),
        psd_tau_prior=(1.0, 300.0),
        data_type="spectroscopy",
    )
    t0 = time.perf_counter()
    res_sub = hf_sub.run(
        "evi",
        n_iterations=50,
        n_samples=6,
        n_posterior_samples=500,
        n_seeds=10,
        verbose=False,
        key=jax.random.PRNGKey(n_sub),
    )
    dt = time.perf_counter() - t0

    sig_s = np.array(res_sub.shared_samples["psd_sigma"])
    tau_s = np.array(res_sub.shared_samples["psd_tau_myr"])
    sw = np.percentile(sig_s, 84) - np.percentile(sig_s, 16)
    tw = np.percentile(tau_s, 84) - np.percentile(tau_s, 16)
    sigma_widths.append(sw)
    tau_widths.append(tw)

    print(
        f"  {n_sub:>2d}  {np.median(sig_s):>9.2f}  "
        f"[{np.percentile(sig_s, 16):.2f}, {np.percentile(sig_s, 84):.2f}]  "
        f"{np.median(tau_s):>9.0f}  "
        f"[{np.percentile(tau_s, 16):.0f}, {np.percentile(tau_s, 84):.0f}]  "
        f"{dt:>5.1f}s"
    )

In [ ]:
# --- FIGURE 6: Posterior width vs N with 1/sqrt(N) reference ---
ns = np.array(N_VALUES, dtype=float)
sigma_widths_arr = np.array(sigma_widths)
tau_widths_arr = np.array(tau_widths)

fig, (ax_sig, ax_tau) = plt.subplots(1, 2, figsize=(10, 4.5))

# sigma scaling
ax_sig.scatter(
    ns,
    sigma_widths_arr,
    s=60,
    color=COLORS["rt"],
    zorder=3,
    label=r"$\sigma_{\rm PS}$ 68% width",
)
n_ref = np.linspace(1.5, 12, 50)
scale_sig = sigma_widths_arr[0] * np.sqrt(ns[0])
ax_sig.plot(
    n_ref,
    scale_sig / np.sqrt(n_ref),
    ls="--",
    color="grey",
    label=r"$\propto 1/\sqrt{N}$",
)
ax_sig.set_xlabel("Number of galaxies $N$")
ax_sig.set_ylabel(r"$\sigma_{\rm PS}$ posterior 68% width")
ax_sig.set_xscale("log")
ax_sig.set_yscale("log")
ax_sig.legend(fontsize=9)
ax_sig.set_title(r"$\sigma_{\rm PS}$ shrinkage")

# tau scaling
ax_tau.scatter(
    ns,
    tau_widths_arr,
    s=60,
    color=COLORS["geovi"],
    zorder=3,
    label=r"$\tau_{\rm PS}$ 68% width",
)
scale_tau = tau_widths_arr[0] * np.sqrt(ns[0])
ax_tau.plot(
    n_ref,
    scale_tau / np.sqrt(n_ref),
    ls="--",
    color="grey",
    label=r"$\propto 1/\sqrt{N}$",
)
ax_tau.set_xlabel("Number of galaxies $N$")
ax_tau.set_ylabel(r"$\tau_{\rm PS}$ posterior 68% width [Myr]")
ax_tau.set_xscale("log")
ax_tau.set_yscale("log")
ax_tau.legend(fontsize=9)
ax_tau.set_title(r"$\tau_{\rm PS}$ shrinkage")

fig.suptitle(
    r"Posterior Shrinkage: $\propto 1/\sqrt{N}$ (spectroscopy)",
    fontsize=11,
)
fig.tight_layout()
plt.savefig(
    os.path.join(FIGDIR, "15_fig06_sqrt_n_scaling.png"),
    dpi=150,
    bbox_inches="tight",
)
plt.show()

## 9. Validation and Diagnostics

Convergence checks and quantitative PSD recovery assessment.

In [ ]:
# --- Convergence diagnostics ---
print("=" * 60)
print("  Individual EVI convergence")
print("=" * 60)
indiv_dict = {f"Galaxy {i}": res for i, res in enumerate(individual_results)}
convergence_table(indiv_dict)

In [ ]:
# --- PSD recovery table ---
print("\n" + "=" * 60)
print("  PSD Recovery: median +/- 68% CI vs truth")
print("=" * 60)
print(f"\n{'Experiment':<25} {'sigma':<25} {'tau [Myr]':<25}")
print("-" * 75)
print(f"{'Truth':<25} {TRUE_SIGMA:<25} {TRUE_TAU:<25}")

for label, s_arr, t_arr in [
    ("Hier. (spectroscopy)", sig_spec, tau_spec),
    ("Hier. (photometry)", sig_phot, tau_phot),
]:
    s_lo, s_med, s_hi = np.percentile(s_arr, [16, 50, 84])
    t_lo, t_med, t_hi = np.percentile(t_arr, [16, 50, 84])
    s_str = f"{s_med:.2f} [{s_lo:.2f}, {s_hi:.2f}]"
    t_str = f"{t_med:.0f} [{t_lo:.0f}, {t_hi:.0f}]"
    print(f"{label:<25} {s_str:<25} {t_str:<25}")

In [ ]:
# --- Posterior predictive check for 4 galaxies ---
print("\nPosterior predictive chi2/dof:")
if result_hier_spec.individual_samples is not None:
    for idx in range(min(4, N_GAL)):
        ind_samp = result_hier_spec.individual_samples[idx]
        flux_obs = np.array(galaxies_spec[idx]["flux_obs"])
        noise = np.array(galaxies_spec[idx]["noise"])

        chi2_vals = []
        n_draw = min(100, len(next(iter(ind_samp.values()))))
        for j in range(n_draw):
            draw = {k: v[j] for k, v in ind_samp.items()}
            draw["sfh_field_psd_sigma"] = TRUE_SIGMA
            draw["sfh_field_psd_tau_myr"] = TRUE_TAU
            try:
                pred = np.array(model_gen.predict_spectrum(draw, WAVE_OBS))
                chi2 = np.sum(((flux_obs - pred) / noise) ** 2) / len(flux_obs)
                chi2_vals.append(chi2)
            except Exception:
                pass

        if chi2_vals:
            chi2_med = np.median(chi2_vals)
            print(f"  Galaxy {idx}: chi2/dof = {chi2_med:.2f}")
else:
    print("  Individual samples not available.")

## Summary

| Experiment | $\sigma_{\rm PS}$ | $\tau_{\rm PS}$ | N | Data | Info/galaxy |
|-----------|-------------------|-----------------|---|------|-------------|
| Individual | wide | unconstrained | 1 | spec (200 pix) | high |
| Hierarchical (spec) | **tight** | **constrained** | 10 | spec (200 pix) | high |
| Hierarchical (phot) | moderate | weakly constrained | 10 | phot (5 bands) | low |

**Key findings:**

1. **Spectroscopy breaks the $\sigma$--$\tau$ degeneracy.** With ~40x more
   data points per galaxy, spectral features (D4000, Balmer lines, UV slope)
   encode burstiness at multiple timescales.

2. **Hierarchical pooling compounds the advantage.** Individual spectroscopic
   fits constrain $\sigma_{\rm PS}$ but not $\tau_{\rm PS}$. The hierarchical
   model recovers both, with posterior width $\propto 1/\sqrt{N}$.

3. **Photometry alone is insufficient for $\tau_{\rm PS}$.** Even with 10
   galaxies, the photometric hierarchical posterior on $\tau_{\rm PS}$ remains
   broad. Spectroscopy is essential for constraining burstiness timescales.

4. **Posterior predictive checks confirm good fits** ($\chi^2/\nu \approx 1$),
   indicating the model correctly captures the spectral information content.

**Implication for surveys:** spectroscopic surveys (DESI, PFS, MOONS) will
enable population-level burstiness constraints that photometric surveys
(LSST, Euclid) cannot match, even with hierarchical modeling.